In [0]:
%sql
CREATE TABLE IF NOT EXISTS hospital_appointments.bronze.batch_reconciliation (
    batch_id STRING,
    previous_silver_count BIGINT,
    input_count BIGINT,
    valid_count BIGINT,
    invalid_count BIGINT,
    duplicate_count BIGINT,
    insert_count BIGINT,
    update_count BIGINT,
    expected_silver_count BIGINT,
    actual_silver_count BIGINT,
    status STRING,
    checked_at TIMESTAMP
);

In [0]:
dbutils.widgets.text("batch_id", "BATCH_02")

dbutils.widgets.dropdown(
    "phase",
    "after_merge",
    ["before_merge", "after_merge"]
)

batch_id = dbutils.widgets.get("batch_id")
phase = dbutils.widgets.get("phase")

print(f"Batch: {batch_id}, phase: {phase}")

In [0]:

dbutils.widgets.text("batch_id", "BATCH_02")
dbutils.widgets.dropdown(
    "phase",
    "before_merge",
    ["before_merge", "after_merge"]
)

batch_id = dbutils.widgets.get("batch_id")
phase = dbutils.widgets.get("phase")

print(f"Batch: {batch_id}, phase: {phase}")

# Table names
bronze_raw = "hospital_appointments.bronze.all_raw_data"
silver_table = "hospital_appointments.silver.silver_data"
valid_table = "hospital_appointments.silver.stg_batch2_valid"
invalid_table = "hospital_appointments.bronze.invalid_batch2"
duplicate_table = "hospital_appointments.bronze.duplicates_batch2"
audit_table = "hospital_appointments.bronze.batch_reconciliation"

batch_sql = batch_id.replace("'", "''")

def count_rows(table, filter_batch=False):
    where_clause = f"WHERE batch_id = '{batch_sql}'" if filter_batch else ""
    return spark.sql(
        f"SELECT COUNT(*) AS n FROM {table} {where_clause}"
    ).first()["n"]

# BEFORE MERGE
if phase == "before_merge":

    input_count = count_rows(bronze_raw, True)
    valid_count = count_rows(valid_table, True)
    invalid_count = count_rows(invalid_table, True)
    duplicate_count = count_rows(duplicate_table, True)
    previous_count = count_rows(silver_table)

    insert_count = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM {valid_table} v
        LEFT ANTI JOIN {silver_table} s
          ON v.appointment_id = s.appointment_id
        WHERE v.batch_id = '{batch_sql}'
    """).first()["n"]

    update_count = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM {valid_table} v
        INNER JOIN {silver_table} s
          ON v.appointment_id = s.appointment_id
        WHERE v.batch_id = '{batch_sql}'
    """).first()["n"]

    expected_count = previous_count + insert_count

    input_check = (
        input_count == valid_count + invalid_count + duplicate_count
    )

    initial_status = "PENDING" if input_check else "FAIL"

    spark.sql(f"""
        MERGE INTO {audit_table} t
        USING (
            SELECT
                '{batch_sql}' AS batch_id,
                {previous_count} AS previous_silver_count,
                {input_count} AS input_count,
                {valid_count} AS valid_count,
                {invalid_count} AS invalid_count,
                {duplicate_count} AS duplicate_count,
                {insert_count} AS insert_count,
                {update_count} AS update_count,
                {expected_count} AS expected_silver_count,
                CAST(NULL AS BIGINT) AS actual_silver_count,
                '{initial_status}' AS status,
                current_timestamp() AS checked_at
        ) s
        ON t.batch_id = s.batch_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"Previous Silver rows: {previous_count}")
    print(f"Input: {input_count}, valid: {valid_count}, invalid: {invalid_count}, duplicates: {duplicate_count}")
    print(f"Inserts: {insert_count}, updates: {update_count}")
    print(f"Expected Silver total after merge: {expected_count}")

    if not input_check:
        raise Exception("FAIL: Input rows do not reconcile.")
